# SignScribe: English (ASL) letter models

Trains, evaluates and exports the two files the web app loads for English letters:
`letters-mlp.json` and `letters-knn.json` (ASL keeps these original names, see `src/recognition/models.ts`).
24 static letters (J and Z move: they need `$1` templates from webcam recordings, not pictures). It runs the repo's own scripts
at commit `c88ee4e`, so features and file format match the app.

**Data: two datasets, merged, 13 people in total.** Each person is one group in the evaluation (leave-one-person-out), so nobody is on both sides of a split.
1. **EWU**: Images of American Sign Language (ASL) Alphabet Gestures (Sowrow, Iftekhar, Prova, Huq; East West University), CC BY 4.0,
   https://data.mendeley.com/datasets/48dg9vhmyk/2. 3 signers. Only the `Type_01_(Raw_Gesture)` photos are used (the other types are skeleton
   renderings). The dataset has no signer field: the signer is **inferred** from the file number (1-500, 501-1000, 1001-1500; checked by eye).
2. **ASL-HG**: ASL-HG American Sign Language Hand Gesture Image Dataset (Pranto, Islam, Akbor, Ghosh, Alam, Chaki, Islam; Bangladesh University of
   Business and Technology), CC BY 4.0, https://data.mendeley.com/datasets/j4y5w2c8w9/1. 10 volunteers (`P1`..`P10`, in the file name), raw smartphone photos.
   Digits and the J/Z pictures are skipped.

**Kaggle settings:** Internet **On**; Accelerator **None**.
- EWU: attach it as a **private** Kaggle dataset (*+ Add Input*), or let the notebook try Mendeley's public API. If you also attach ASL-HG,
  set `INPUT_PATH` to the EWU dataset's own folder so step 3 does not walk through both.
- ASL-HG: downloaded automatically from Mendeley (only the raw zip). If that fails, download `ASL_Raw_Images.zip` in your browser, upload it as a
  private Kaggle dataset and attach it: step 3b finds it by itself.

**Slow step:** MediaPipe on every picture (about 60,000). `MAX_PER_CLASS_PER_SIGNER` keeps a random subset (fixed seed) of the EWU pictures.

**The layout of ASL-HG (`<class>/P<k>_<class>_<n>.jpg`) comes from a third-party description, not from the dataset's own page.** Step 3b prints what it
finds and stops with the first file names if it does not match.

## Before you share anything from this notebook
- **Keep this notebook and its output private.** The `*-knn.json` file stores real hand landmark samples
  (the repo never commits it), and the takes file holds landmarks of every picture.
- **Only quote numbers that a script wrote** (`MODEL_RESULTS.md`, the `reports/` files). These are numbers on **dataset pictures**
  where MediaPipe found a hand, not on a webcam: real-camera accuracy is still **NOT MEASURED**.
- `MODEL_RESULTS.md` in the output is the repo's file with this run appended by `ml/evaluate.py`; it replaces `docs/MODEL_RESULTS.md`
  in a branch (then update section 6 of `docs/PROJECT_STATUS.md`; `npm run check` compares the two).


## 1. Settings

In [4]:
REPO_URL = "https://github.com/Youssef-Adel91/SignScribe.git"
REPO_COMMIT = "c88ee4e"      # main on 2026-11-01; set to None for the newest main
NOTEBOOK_NAME = "signscribe_asl_letters"

# ---- source 1: East West University (EWU), 3 signers (Kaggle input or Mendeley download) ----
INPUT_PATH = None                  # None = whatever is attached, else download. With ASL-HG attached too, set the EWU dataset's own folder,
                                   # e.g. "/kaggle/input/datasets/davidnashaat1910/asl-alphabet"
MENDELEY_ID, MENDELEY_VERSION = "48dg9vhmyk", 2
EWU_NAME = "ASL Alphabet Gestures (East West University)"
EWU_VERSION = "Mendeley 48dg9vhmyk v2, Type_01 only, signer = file-number block of 500 (inferred)"

# ---- source 2: ASL-HG (Bangladesh University of Business and Technology), 10 volunteers, CC BY 4.0 ----
HG_INPUT_PATH = None               # None = look in /kaggle/input for the raw zip or pictures, else download from Mendeley
HG_MENDELEY_ID, HG_MENDELEY_VERSION = "j4y5w2c8w9", 1
HG_NAME = "ASL-HG American Sign Language Hand Gesture Image Dataset"
HG_VERSION = "Mendeley j4y5w2c8w9 v1, raw images, signer = P1..P10 from the file name"

# ---- the merged data set that is evaluated and exported ----
DATASET_NAME = "ASL letters, EWU plus ASL-HG merged"
DATASET_VERSION = f"{EWU_VERSION}; {HG_VERSION}"

MAX_PER_CLASS_PER_SIGNER = 500     # pictures per letter per signer (random, seed 0); None = all
SEED = 0
SIGNING_HAND = "right"             # unknown for both datasets; mirror augmentation covers the other hand
SHARED_CLASSES = False             # True only if evaluation stops because a person has no pictures of some letter

REIMPORT = False
KEEP_TAKES = False


## 2. Setup

In [5]:
# ---- setup: paths, a helper that runs a command and streams its output ----
import hashlib, json, os, random, re, shutil, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "work"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp/signscribe")
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

# only for checking this notebook itself with generated hands: never set it for a real run
PIPELINE_TEST = os.environ.get("SIGNSCRIBE_PIPELINE_TEST") == "1"
if PIPELINE_TEST:
    print("PIPELINE TEST MODE: generated data, the results mean nothing")

def run(cmd, cwd=None):
    """Runs a command, prints its output while it runs, stops the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}: {' '.join(cmd[:3])} ...")

def sha256_of(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n]

print("work folder (kept as notebook output):", WORK)
print("scratch folder (not kept):", SCRATCH)

work folder (kept as notebook output): /kaggle/working
scratch folder (not kept): /tmp/signscribe


In [6]:
# ---- get the SignScribe code (the same ml/ scripts the team runs) + MediaPipe + the hand model ----
REPO = Path(os.environ.get("SIGNSCRIBE_REPO_DIR", SCRATCH / "SignScribe"))
if not (REPO / ".git").exists():
    run(["git", "clone", "--quiet", REPO_URL, REPO])
if REPO_COMMIT and not os.environ.get("SIGNSCRIBE_REPO_DIR"):
    run(["git", "checkout", "--quiet", REPO_COMMIT], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print("SignScribe commit:", COMMIT)

try:
    import mediapipe  # noqa: F401
except ImportError:
    run([sys.executable, "-m", "pip", "install", "--quiet", "mediapipe>=0.10.14,<0.11", "numpy"])
import mediapipe
print("mediapipe", mediapipe.__version__)

# the same file and URL as tools/get-models.mjs (npm run models)
HAND_MODEL = REPO / "public" / "models" / "hand_landmarker.task"
HAND_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
if not HAND_MODEL.exists() and not PIPELINE_TEST:
    HAND_MODEL.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL)
if HAND_MODEL.exists():
    print("hand model:", HAND_MODEL.name, f"{HAND_MODEL.stat().st_size/1e6:.1f} MB, sha256", sha256_of(HAND_MODEL))

$ git checkout --quiet c88ee4e
SignScribe commit: c88ee4e
mediapipe 0.10.35
hand model: hand_landmarker.task 7.8 MB, sha256 fbc2a30080c3c557


In [7]:
# ---- dataset helpers: use an attached Kaggle dataset, else download from Mendeley Data ----
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def extract_all_zips(folder):
    """Extracts every .zip under `folder` next to itself (zips inside zips too), then deletes the extracted zip copy."""
    done = set()
    while True:
        zips = [z for z in Path(folder).rglob("*.zip") if z not in done]
        if not zips:
            return
        for z in zips:
            target = z.with_suffix("")
            print("extracting", z.relative_to(folder), flush=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
            done.add(z)
            z.unlink()  # only copies in the scratch folder are ever deleted

def mendeley_download(dataset_id, version, dest, only=None):
    """Downloads every file of a public Mendeley Data dataset through its public API."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    def get_json(url):
        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "signscribe-notebook"})
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.load(r)
    base = f"https://data.mendeley.com/public-api/datasets/{dataset_id}"
    folder_ids = ["root"]
    try:
        folder_ids += [f["id"] for f in get_json(f"{base}/folders/{version}")]
    except Exception as e:
        print("(no folder list:", e, ")")
    got = 0
    for fid in folder_ids:
        for f in get_json(f"{base}/files?folder_id={fid}&version={version}"):
            if only and only.lower() not in f["filename"].lower():
                continue
            url = (f.get("content_details") or {}).get("download_url")
            if not url:
                continue
            out = dest / f["filename"]
            size = (f.get("content_details") or {}).get("size") or f.get("size")
            print(f"downloading {f['filename']} ({(size or 0)/1e6:.0f} MB)", flush=True)
            urllib.request.urlretrieve(url, out)
            got += 1
    if not got:
        raise RuntimeError("the Mendeley API listed no files")

def get_dataset(dest, input_path, dataset_id, version, page):
    """Returns a scratch folder with the dataset's files, zips extracted. Never writes into /kaggle/input."""
    dest = Path(dest)
    marker = dest / ".ready"
    if marker.exists():
        print("dataset already prepared in", dest)
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(input_path) if input_path else None
    if src is None and Path("/kaggle/input").exists() and any(Path("/kaggle/input").iterdir()):
        src = Path("/kaggle/input")
    if src is not None:
        print("using the attached files in", src)
        for z in src.rglob("*.zip"):  # zips are copied so they can be extracted; pictures are linked, not copied
            if z.name.lower().startswith("asl_raw"):  # the ASL-HG zip belongs to step 3b, not to EWU
                continue
            rel = z.relative_to(src)
            (dest / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(z, dest / rel)
        link = dest / "attached"
        if not link.exists():
            link.symlink_to(src.resolve(), target_is_directory=True)
    else:
        try:
            mendeley_download(dataset_id, version, dest)
        except Exception as e:
            raise RuntimeError(
                f"Automatic download failed ({e}).\n"
                f"Download the dataset in your browser from {page} , upload the zip as a PRIVATE Kaggle dataset, "
                f"attach it with '+ Add Input' (right panel) and run this cell again. Or set INPUT_PATH."
            ) from e
    extract_all_zips(dest)
    marker.write_text("ok")
    return dest

def images_under(folder):
    """Every picture under `folder`, following the link to an attached dataset."""
    found = []
    for d, _, files in os.walk(folder, followlinks=True):
        found += [Path(d) / f for f in files if Path(f).suffix.lower() in IMAGE_SUFFIXES]
    return sorted(found)

## 3. EWU: get the dataset, find its layout (class folder, signer), take a subset

In [8]:
RAW = get_dataset(SCRATCH / "asl_raw", INPUT_PATH, MENDELEY_ID, MENDELEY_VERSION,
                  "https://data.mendeley.com/datasets/48dg9vhmyk/2")
sys.path.insert(0, str(REPO / "ml"))
from signscribe_ml.alphabet import load_alphabet
ALPHA = load_alphabet("asl")
DATASET_CLASSES = {chr(c) for c in range(ord("a"), ord("z") + 1)} | {"del", "space", "nothing"}

pics = images_under(RAW)
print("pictures found:", len(pics))
rows = []  # (relative folder parts, class folder index, file)
for p in pics:
    parts = p.relative_to(RAW).parts[:-1]
    cls_idx = max((i for i, x in enumerate(parts) if x.strip().lower() in DATASET_CLASSES), default=None)
    rows.append((parts, cls_idx, p))
no_class = sum(1 for r in rows if r[1] is None)
print("pictures with no class folder (A..Z, Del, Space) in their path:", no_class)
rows = [r for r in rows if r[1] is not None]
assert rows, "no class folders found: look at the layout below and adapt this cell"

# only the real photos: Type_02..05 are renderings (keypoints/skeletons), not pictures of hands
rows = [r for r in rows if any(x.startswith("Type_01") for x in r[0])]
print("kept only Type_01_(Raw_Gesture):", len(rows), "pictures")
assert rows, "no Type_01 folder found: look at the layout and adapt this cell"

depth_values = {}
for parts, ci, _ in rows:
    for d, x in enumerate(parts):
        if d != ci and x.strip().lower() not in DATASET_CLASSES:
            depth_values.setdefault(d, set()).add(x)
print("folder levels other than the class folder (level: number of different names, examples):")
for d, v in sorted(depth_values.items()):
    print(f"  {d}: {len(v)}  {sorted(v)[:6]}")

def signer_of(parts, p):
    # the dataset numbers pictures per letter: 1-500 signer 1, 501-1000 signer 2, 1001-1500 signer 3
    # (checked by eye on the letter A: person and skin tone change exactly at 500/501 and 1000/1001)
    m = re.search(r"_(\d+)\.\w+$", p.name)
    return f"p{(int(m.group(1)) - 1) // 500 + 1}" if m else None

# sanity check: every letter should have 500 pictures per signer
_chk = {}
for parts, ci, p in rows:
    s = signer_of(parts, p)
    key = (parts[ci].upper(), s)
    _chk[key] = _chk.get(key, 0) + 1
odd = {k: v for k, v in _chk.items() if v != 500}
print("letter/signer groups that do NOT have 500 pictures:", odd or "none")

by = {}
skipped = {}
for parts, ci, p in rows:
    label = ALPHA.resolve(parts[ci])
    if label is None or ALPHA.kind_of(label) != "static":
        skipped[parts[ci]] = skipped.get(parts[ci], 0) + 1
        continue
    s = signer_of(parts, p)
    if s is None:
        skipped["(no signer)"] = skipped.get("(no signer)", 0) + 1
        continue
    by.setdefault((s, label), []).append(p)
signers = sorted({s for s, _ in by})
SIGNER_ID = {s: f"s{i+1}" for i, s in enumerate(signers)}
print("signers:", {k: v for k, v in SIGNER_ID.items()})
print("not imported (not a static letter of the app's ASL alphabet, or no signer):", skipped)
assert len(signers) >= 2, "leave-one-signer-out needs at least 2 signers"

# a flat folder of links named <signer>_<LETTER>_<n>.<ext>; the import reads letter and signer from the name
rng = random.Random(SEED)
STAGE = SCRATCH / "asl_stage"
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir()
count = {}
for (s, label), files in sorted(by.items()):
    files = sorted(files)
    if MAX_PER_CLASS_PER_SIGNER and len(files) > MAX_PER_CLASS_PER_SIGNER:
        files = sorted(rng.sample(files, MAX_PER_CLASS_PER_SIGNER))
    for n, f in enumerate(files):
        (STAGE / f"{SIGNER_ID[s]}_{label}_{n}{f.suffix.lower()}").symlink_to(f.resolve())
    count[label] = count.get(label, 0) + len(files)
print(f"staged {sum(count.values())} pictures, {len(count)} letters (expected 24):")
print("  ".join(f"{k}:{v}" for k, v in sorted(count.items())))

using the attached files in /kaggle/input
pictures found: 282000
pictures with no class folder (A..Z, Del, Space) in their path: 20000
kept only Type_01_(Raw_Gesture): 42000 pictures
folder levels other than the class folder (level: number of different names, examples):
  0: 1  ['attached']
  1: 1  ['datasets']
  2: 1  ['davidnashaat1910']
  3: 1  ['asl-alphabet']
  4: 1  ['Images of American Sign Language (ASL) Alphabet Gestures']
  5: 1  ['Root']
  6: 1  ['Root']
  7: 1  ['Type_01_(Raw_Gesture)']
letter/signer groups that do NOT have 500 pictures: none
signers: {'p1': 's1', 'p2': 's2', 'p3': 's3'}
not imported (not a static letter of the app's ASL alphabet, or no signer): {'DEL': 1500, 'J': 1500, 'SPACE': 1500, 'Z': 1500}
staged 36000 pictures, 24 letters (expected 24):
A:1500  B:1500  C:1500  D:1500  E:1500  F:1500  G:1500  H:1500  I:1500  K:1500  L:1500  M:1500  N:1500  O:1500  P:1500  Q:1500  R:1500  S:1500  T:1500  U:1500  V:1500  W:1500  X:1500  Y:1500


## 3b. ASL-HG: get the dataset (10 volunteers, the signer is in the file name)

In [9]:
# ---- 3b. ASL-HG: 10 volunteers; the signer (P1..P10) is in the file name ----
import collections
HG_PIC = re.compile(r"^P(\d+)_([A-Za-z0-9]+)_(\d+)\.\w+$", re.I)

def get_hg():
    """The folder with the ASL-HG raw pictures: attached files if there are any, else downloaded from Mendeley Data."""
    dest = SCRATCH / "hg_raw"
    if (dest / ".ready").exists():
        print("ASL-HG already prepared in", dest)
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(HG_INPUT_PATH) if HG_INPUT_PATH else (Path("/kaggle/input") if Path("/kaggle/input").exists() else None)
    if src is not None:
        print("looking for attached ASL-HG files in", src, "...", flush=True)
        z = next(iter(sorted(src.rglob("ASL_Raw*.zip"))), None)
        if z is not None:
            print("using the attached", z)
            shutil.copy2(z, dest / z.name)
            extract_all_zips(dest)
            (dest / ".ready").write_text("ok")
            return dest
        cands = []                                         # folders that hold P<k>_<class>_<n> pictures
        for d, _, files in os.walk(src, followlinks=True):
            if any(HG_PIC.match(f) for f in files):
                cands.append(Path(d))
        # only the RAW photos: the dataset also has a processed version (MediaPipe-processed, split into train/val/test)
        # (judged by the path below the attached folder, so a user or dataset name cannot fool it)
        raw = [c for c in cands if "raw" in str(c.relative_to(src)).lower() and "processed" not in str(c.relative_to(src)).lower()]
        if raw:
            root = Path(os.path.commonpath([str(c) for c in raw]))     # read-only: nothing is written there
            print("using the attached RAW ASL-HG pictures under", root)
            return root
        if cands:
            print("the attached ASL-HG has no raw pictures (only e.g.", cands[0], ") -> not used, downloading the raw zip from Mendeley instead")
    try:
        mendeley_download(HG_MENDELEY_ID, HG_MENDELEY_VERSION, dest, only="raw")   # only the raw pictures, not the processed zip
    except Exception as e:
        raise RuntimeError(
            f"Automatic download of ASL-HG failed ({e}).\n"
            f"Download ASL_Raw_Images.zip in your browser from https://data.mendeley.com/datasets/{HG_MENDELEY_ID}/{HG_MENDELEY_VERSION} , "
            f"upload it as a PRIVATE Kaggle dataset, attach it with '+ Add Input' and run this cell again (or set HG_INPUT_PATH)."
        ) from e
    extract_all_zips(dest)
    (dest / ".ready").write_text("ok")
    return dest

HG_ROOT = get_hg()
hg_pics = images_under(HG_ROOT)
print("ASL-HG pictures found:", len(hg_pics))

rows_hg, not_letters, mismatch = [], collections.Counter(), 0
for p in hg_pics:
    m = HG_PIC.match(p.name)
    cls = p.parent.name.strip()
    if not m:
        not_letters["(file name is not P<k>_<class>_<n>)"] += 1
        continue
    if not (len(cls) == 1 and cls.isalpha()):          # digits and anything that is not a single letter
        not_letters[cls] += 1
        continue
    if m.group(2).upper() != cls.upper():
        mismatch += 1
    rows_hg.append((int(m.group(1)), cls.upper(), p))
if not rows_hg:
    print("No picture matched P<k>_<class>_<n>.jpg inside a one-letter folder. The first 10 files:")
    for p in hg_pics[:10]:
        print("  ", p.relative_to(HG_ROOT))
    raise AssertionError("ASL-HG has a different layout than expected: send me these lines and I adapt HG_PIC / the folder rule")
print("letter pictures:", len(rows_hg), "| skipped (digits and others):", dict(not_letters),
      "| folder letter differs from file-name letter:", mismatch)

by_hg, skipped_hg = {}, collections.Counter()
for n, cls, p in rows_hg:
    label = ALPHA.resolve(cls)
    if label is None or ALPHA.kind_of(label) != "static":      # J and Z are motion letters: the app records templates for them
        skipped_hg[cls] += 1
        continue
    by_hg.setdefault((n, label), []).append(p)
hg_people = sorted({n for n, _ in by_hg})
HG_SIGNER_ID = {n: f"s{len(SIGNER_ID) + i + 1}" for i, n in enumerate(hg_people)}   # after the EWU signers: s4, s5, ...
print("ASL-HG volunteers:", [f"P{n}" for n in hg_people], "| not imported (not a static letter):", dict(skipped_hg))
odd = {(f"P{n}", l): len(fs) for (n, l), fs in sorted(by_hg.items()) if len(fs) != 100}
print("volunteer/letter groups that do NOT have 100 pictures:", odd or "none")
assert len(hg_people) >= 2, "expected several volunteers: check the file names"

rng_hg = random.Random(SEED)
STAGE_HG = SCRATCH / "hg_stage"
if STAGE_HG.exists():
    shutil.rmtree(STAGE_HG)
STAGE_HG.mkdir()
count_hg = collections.Counter()
for (n, label), files in sorted(by_hg.items()):
    files = sorted(files)
    if MAX_PER_CLASS_PER_SIGNER and len(files) > MAX_PER_CLASS_PER_SIGNER:
        files = sorted(rng_hg.sample(files, MAX_PER_CLASS_PER_SIGNER))
    for k, f in enumerate(files):
        (STAGE_HG / f"{HG_SIGNER_ID[n]}_{label}_{k}{f.suffix.lower()}").symlink_to(f.resolve())
    count_hg[label] += len(files)
print(f"staged {sum(count_hg.values())} ASL-HG pictures, {len(count_hg)} letters (expected 24 letters x 10 volunteers x 100 = 24000):")
print("  ".join(f"{k}:{v}" for k, v in sorted(count_hg.items())))

looking for attached ASL-HG files in /kaggle/input ...
using the attached RAW ASL-HG pictures under /kaggle/input/datasets/davidnashaat1910/asl-hg/ASL-HG American Sign Language Hand Gesture Image D/ASL_HG_36000/ASL_Raw_Images/asl_dataset
ASL-HG pictures found: 36000
letter pictures: 26000 | skipped (digits and others): {'0': 1000, '1': 1000, '2': 1000, '3': 1000, '4': 1000, '5': 1000, '6': 1000, '7': 1000, '8': 1000, '9': 1000} | folder letter differs from file-name letter: 0
ASL-HG volunteers: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10'] | not imported (not a static letter): {'J': 1000, 'Z': 1000}
volunteer/letter groups that do NOT have 100 pictures: none
staged 24000 ASL-HG pictures, 24 letters (expected 24 letters x 10 volunteers x 100 = 24000):
A:1000  B:1000  C:1000  D:1000  E:1000  F:1000  G:1000  H:1000  I:1000  K:1000  L:1000  M:1000  N:1000  O:1000  P:1000  Q:1000  R:1000  S:1000  T:1000  U:1000  V:1000  W:1000  X:1000  Y:1000


## 4. Pictures → hand landmarks (MediaPipe) → takes file (both sources, joined)

In [10]:
TAKES_EWU = SCRATCH / "asl-ewu.jsonl"
TAKES_HG = SCRATCH / "asl-hg.jsonl"
TAKES = SCRATCH / "asl.jsonl"                      # both sources together: what every later step reads
for f in (TAKES_EWU, TAKES_HG, TAKES):             # always rebuild
    f.unlink(missing_ok=True)
    f.with_suffix(".import.md").unlink(missing_ok=True)

assert len(list(STAGE.iterdir())) > 30000, f"EWU STAGE has {len(list(STAGE.iterdir()))} pictures: run the EWU dataset cell first"
assert len(list(STAGE_HG.iterdir())) > 10000, f"ASL-HG STAGE has {len(list(STAGE_HG.iterdir()))} pictures: run the ASL-HG cell (3b) first"

for stage, out, name, source in [(STAGE, TAKES_EWU, EWU_NAME, "asl-ewu"), (STAGE_HG, TAKES_HG, HG_NAME, "asl-hg")]:
    run([sys.executable, REPO / "ml/import_images.py", "--images", stage, "--out", out, "--overwrite",
         "--alphabet", "asl", "--dataset-name", name, "--source", source, "--hand", SIGNING_HAND,
         "--label-regex", r"^s\d+_([A-Z])_\d+\.\w+$", "--person-regex", r"^(s\d+)_",
         "--model", HAND_MODEL], cwd=REPO)

with open(TAKES, "w", encoding="utf-8") as fh:     # a takes file is plain JSON lines, so joining two is safe (ids differ by source)
    for f in (TAKES_EWU, TAKES_HG):
        fh.write(f.read_text(encoding="utf-8"))
print("takes:", sum(1 for _ in open(TAKES)), " sha256", sha256_of(TAKES))

import collections
by_person, src_of = collections.defaultdict(collections.Counter), {}
for line in open(TAKES):
    t = json.loads(line)
    by_person[t["person"]][t["label"]] += 1
    src_of[t["person"]] = t["source"]
print("one line per person: takes kept, letters covered, fewest and most takes for one letter")
for p in sorted(by_person, key=lambda s: int(s[1:])):
    c = by_person[p]
    lo = min(c, key=c.get)
    print(f"  {p:4} {src_of[p]:8} takes {sum(c.values()):5}  letters {len(c):2}  fewest {c[lo]:4} ({lo})  most {max(c.values()):4}")

$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/import_images.py --images /tmp/signscribe/asl_stage --out /tmp/signscribe/asl-ewu.jsonl --overwrite --alphabet asl --dataset-name ASL Alphabet Gestures (East West University) --source asl-ewu --hand right --label-regex ^s\d+_([A-Z])_\d+\.\w+$ --person-regex ^(s\d+)_ --model /tmp/signscribe/SignScribe/public/models/hand_landmarker.task
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1793704812.602344     213 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793704812.623270     213 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793704827.208769     213 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECT

## 5. Honest evaluation: leave-one-person-out
13 people (3 EWU + 10 ASL-HG): each fold trains on the others and tests on one person who was never in training. The merged run is the result to quote.
Each source is then evaluated alone with the same method, so you can see what the merge changed.

In [11]:
OUT = WORK / "signscribe-asl-output"
REPORTS, MODELS = OUT / "reports", OUT / "models"
test_flag = ["--allow-synthetic"] if PIPELINE_TEST else []
results_md = [] if PIPELINE_TEST else ["--results-md", REPO / "docs/MODEL_RESULTS.md"]
subset = f", subset of {MAX_PER_CLASS_PER_SIGNER} per letter per signer (seed {SEED})" if MAX_PER_CLASS_PER_SIGNER else ""
version = DATASET_VERSION + subset

# start clean: the repo's committed results file (no section of an earlier run) and no old reports
subprocess.run(["git", "checkout", "--", "docs/MODEL_RESULTS.md"], cwd=REPO, check=True)
shutil.rmtree(REPORTS, ignore_errors=True)
for d in (REPORTS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

def evaluate(takes, name, ver, out_dir, ablation):
    out_dir.mkdir(parents=True, exist_ok=True)
    run([sys.executable, REPO / "ml/evaluate.py", "--data", takes, "--alphabet", "asl",
         "--dataset-name", name, "--dataset-version", ver, "--augment", "all",
         *(["--augment-ablation"] if ablation else []), *(["--shared-classes"] if SHARED_CLASSES else []), "--out-dir", out_dir, *results_md, *test_flag], cwd=REPO)
    return max(out_dir.glob("eval-*.json"), key=lambda p: p.stat().st_mtime)

# 1. both sources together: the result to quote, and the evaluation the models are exported with
EVAL_JSON = evaluate(TAKES, DATASET_NAME, version, REPORTS, True)
print("evaluation report:", EVAL_JSON.with_suffix(".md").name)

# 2. each source alone with the same method, to see what the merge changed
for takes, name, ver, folder in [(TAKES_EWU, EWU_NAME, EWU_VERSION + subset, "ewu-only"),
                                 (TAKES_HG, HG_NAME, HG_VERSION + subset, "hg-only")]:
    evaluate(takes, name, ver, REPORTS / folder, False)

$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/evaluate.py --data /tmp/signscribe/asl.jsonl --alphabet asl --dataset-name ASL letters, EWU plus ASL-HG merged --dataset-version Mendeley 48dg9vhmyk v2, Type_01 only, signer = file-number block of 500 (inferred); Mendeley j4y5w2c8w9 v1, raw images, signer = P1..P10 from the file name, subset of 500 per letter per signer (seed 0) --augment all --augment-ablation --out-dir /kaggle/working/signscribe-asl-output/reports --results-md /tmp/signscribe/SignScribe/docs/MODEL_RESULTS.md
## with augmentation

### Static letters: leave-one-group-out evaluation

- command: `python ml/evaluate.py --data /tmp/signscribe/asl.jsonl --alphabet asl --dataset-name ASL letters, EWU plus ASL-HG merged --dataset-version Mendeley 48dg9vhmyk v2, Type_01 only, signer = file-number block of 500 (inferred); Mendeley j4y5w2c8w9 v1, raw images, signer = P1..P10 from the file name, subset of 500 per letter per signer (seed 0) --augment all --augment-ablation --out-dir

## 6. Confidence thresholds (a table to choose from by hand)

In [12]:
run([sys.executable, REPO / "ml/calibrate.py", "--data", TAKES, "--alphabet", "asl",
     "--dataset-name", DATASET_NAME, "--dataset-version", version, "--out-dir", REPORTS, *test_flag], cwd=REPO)

$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/calibrate.py --data /tmp/signscribe/asl.jsonl --alphabet asl --dataset-name ASL letters, EWU plus ASL-HG merged --dataset-version Mendeley 48dg9vhmyk v2, Type_01 only, signer = file-number block of 500 (inferred); Mendeley j4y5w2c8w9 v1, raw images, signer = P1..P10 from the file name, subset of 500 per letter per signer (seed 0) --out-dir /kaggle/working/signscribe-asl-output/reports
# Confidence calibration: ASL letters, EWU plus ASL-HG merged Mendeley 48dg9vhmyk v2, Type_01 only, signer = file-number block of 500 (inferred); Mendeley j4y5w2c8w9 v1, raw images, signer = P1..P10 from the file name, subset of 500 per letter per signer (seed 0)


- dataset sha256 `2aeac81a780b7277ae2246b787d22e291c1ed4bdfa46d07f95074c40523b7cc5`
- alphabet `asl`, augmentation (training side only): rotate+scale+mirror x2 copies per frame, k=5, seed 0
- command: `python ml/calibrate.py --data /tmp/signscribe/asl.jsonl --alphabet asl --dataset-name ASL lette

## 7. Export `letters-mlp.json` and `letters-knn.json` (trained on all signers) and validate them

In [13]:
common = ["--alphabet", "asl", "--augment", "all", "--seed", "0", *test_flag]
run([sys.executable, REPO / "ml/export_mlp.py", "--data", TAKES, "--out", MODELS / "letters-mlp.json",
     "--evaluation-json", EVAL_JSON, *common], cwd=REPO)
run([sys.executable, REPO / "ml/train_knn.py", "--data", TAKES, "--out", MODELS / "letters-knn.json",
     "--max-per-class", "300", *common], cwd=REPO)
labels = []
for f in (MODELS / "letters-mlp.json", MODELS / "letters-knn.json"):
    run([sys.executable, REPO / "ml/validate_model.py", f, "--alphabet", "asl"], cwd=REPO)
    labels.append(json.loads(f.read_text())["labels"])
assert labels[0] == labels[1], "the two files do not list the same letters in the same order"
print("OK: both files list the same", len(labels[0]), "letters:", " ".join(labels[0]))

$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/export_mlp.py --data /tmp/signscribe/asl.jsonl --out /kaggle/working/signscribe-asl-output/models/letters-mlp.json --evaluation-json /kaggle/working/signscribe-asl-output/reports/eval-ASL_letters__EWU_plus_ASL-HG_merged-2026-11-03.json --alphabet asl --augment all --seed 0
wrote /kaggle/working/signscribe-asl-output/models/letters-mlp.json (66 kB, 24 labels) and /kaggle/working/signscribe-asl-output/models/letters-mlp.report.md
$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/train_knn.py --data /tmp/signscribe/asl.jsonl --out /kaggle/working/signscribe-asl-output/models/letters-knn.json --max-per-class 300 --alphabet asl --augment all --seed 0
wrote /kaggle/working/signscribe-asl-output/models/letters-knn.json (7200 samples, 24 letters) and /kaggle/working/signscribe-asl-output/models/letters-knn.report.md
$ /usr/bin/python3 /tmp/signscribe/SignScribe/ml/validate_model.py /kaggle/working/signscribe-asl-output/models/letters-mlp.json --a

## 8. Package the output

In [14]:
# ---- collect everything the team needs into one folder and one zip (the notebook's output) ----
for p in [REPO / "docs" / "MODEL_RESULTS.md", TAKES_EWU.with_suffix(".import.md"), TAKES_HG.with_suffix(".import.md")]:
    if p.exists():
        shutil.copy2(p, OUT / ("MODEL_RESULTS.md" if p.name == "MODEL_RESULTS.md" else "reports/" + p.name))
if KEEP_TAKES and TAKES.exists():
    shutil.copy2(TAKES, OUT / TAKES.name)
info = {
    "notebook": NOTEBOOK_NAME,
    "signscribe_commit": COMMIT,
    "mediapipe": mediapipe.__version__,
    "hand_model_sha256_16": sha256_of(HAND_MODEL) if HAND_MODEL.exists() else None,
    "dataset": DATASET_NAME, "dataset_version": DATASET_VERSION,
    "takes_sha256_16": sha256_of(TAKES),
    "takes_ewu_sha256_16": sha256_of(TAKES_EWU), "takes_hg_sha256_16": sha256_of(TAKES_HG),
    "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "pipeline_test": PIPELINE_TEST,
}
(OUT / "RUN_INFO.json").write_text(json.dumps(info, indent=1))
archive = shutil.make_archive(str(WORK / OUT.name), "zip", OUT)
print("\nOutput folder:", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size/1e3:.0f} kB)")
print("zip:", archive)


Output folder: /kaggle/working/signscribe-asl-output
  MODEL_RESULTS.md  (35 kB)
  RUN_INFO.json  (1 kB)
  models/letters-knn.json  (3347 kB)
  models/letters-knn.report.md  (2 kB)
  models/letters-mlp.json  (66 kB)
  models/letters-mlp.report.md  (1 kB)
  reports/asl-ewu.import.md  (2 kB)
  reports/asl-hg.import.md  (2 kB)
  reports/calibration-ASL_letters__EWU_plus_ASL-HG_merged-2026-11-03.json  (6 kB)
  reports/calibration-ASL_letters__EWU_plus_ASL-HG_merged-2026-11-03.md  (4 kB)
  reports/eval-ASL_letters__EWU_plus_ASL-HG_merged-2026-11-03.json  (22 kB)
  reports/eval-ASL_letters__EWU_plus_ASL-HG_merged-2026-11-03.md  (9 kB)
  reports/ewu-only/eval-ASL_Alphabet_Gestures__East_West_University_-2026-11-03.json  (12 kB)
  reports/ewu-only/eval-ASL_Alphabet_Gestures__East_West_University_-2026-11-03.md  (5 kB)
  reports/hg-only/eval-ASL-HG_American_Sign_Language_Hand_Gesture_Image_Dataset-2026-11-03.json  (11 kB)
  reports/hg-only/eval-ASL-HG_American_Sign_Language_Hand_Gesture_Image_

## 9. Putting the models into the app (on your computer, in a branch)
1. Download the output zip. Copy `models/letters-mlp.json` and `models/letters-knn.json` into `public/models/`.
   **Never commit the k-NN file.** Whether to commit the MLP file is a team decision.
2. Record the run: replace `docs/MODEL_RESULTS.md` with the zip's copy, update `docs/PROJECT_STATUS.md` section 6,
   `docs/CREDITS.md` (both datasets' citations and licences: EWU Mendeley 48dg9vhmyk, ASL-HG Mendeley j4y5w2c8w9 / DOI 10.17632/j4y5w2c8w9.1, both CC BY 4.0), run `npm run check`, open a PR.
3. Thresholds by hand in `src/recognition/tuning.ts` from `reports/*calibration*`, with a test.
4. Still missing for English: J and Z (`tools/dollarpy/record_templates.py` on a webcam), and the real-webcam test.